## Save and reload a GMM for a quick demonstration

In [ ]:
# SAVE DEMO GMM MODEL FOR QUICK DEMONSTRATION

demo_scheme = "gene_disjoint"
demo_feature_set = "D"

# Use only the training portion of the first gene-disjoint fold.
demo_fold = folds[demo_scheme][0]
train_idx = demo_fold["train"]

feature_names = list(feature_sets[demo_feature_set])
X_raw = df[feature_names].to_numpy(dtype=float)
y = y_all.copy()

X_train_raw = X_raw[train_idx]
y_train = y[train_idx]

# Learn imputation and standardization values from training rows only.
impute_median = np.nanmedian(X_train_raw, axis=0)
impute_median = np.where(np.isnan(impute_median), 0.0, impute_median)
X_train_imputed = np.where(np.isnan(X_train_raw), impute_median, X_train_raw)

train_mean = X_train_imputed.mean(axis=0)
train_std = X_train_imputed.std(axis=0)
train_std[train_std < 1e-12] = 1.0
X_train = (X_train_imputed - train_mean) / train_std

# Fit and select one mixture for each class using BIC.
class_models = {}
for class_value, class_name in [(0, "benign"), (1, "pathogenic")]:
    X_class = X_train[y_train == class_value]
    candidates = []

    for covariance_type in COV_TYPES:
        for K in K_GRID:
            fitted_model, fit_info = fit_mixture(
                X_class,
                K,
                covariance_type,
                [SEED, 999, class_value, K]
            )

            bic = (
                -2 * fitted_model["ll"] * len(X_class)
                + n_params(K, X_train.shape[1], covariance_type)
                * np.log(len(X_class))
            )

            candidates.append({
                "bic": bic,
                "K": K,
                "covariance_type": covariance_type,
                "model": fitted_model
            })

    best = min(candidates, key=lambda item: item["bic"])
    class_models[class_name] = {
        "K": best["K"],
        "covariance_type": best["covariance_type"],
        "model": best["model"],
        "bic": best["bic"]
    }

demo_gmm_bundle = {
    "model_type": "BIC-selected Gaussian mixture classifier",
    "feature_set": demo_feature_set,
    "feature_names": feature_names,
    "validation_scheme": demo_scheme,
    "repeat": demo_fold["repeat"],
    "fold": demo_fold["fold"],
    "impute_median": impute_median,
    "train_mean": train_mean,
    "train_std": train_std,
    "class_models": class_models,
    "pathogenic_prior": float(y_train.mean()),
    "log_prior_ratio": float(np.log(y_train.mean() / (1 - y_train.mean()))),
    "regularization": REG_COVAR
}

demo_model_path = OUT_DIR / "demo_gmm_setD_gene_disjoint.joblib"
joblib.dump(demo_gmm_bundle, demo_model_path)

print("Saved:", demo_model_path)
print("File size in bytes:", demo_model_path.stat().st_size)
print("Selected benign model:", class_models["benign"]["covariance_type"],
      "K=", class_models["benign"]["K"])
print("Selected pathogenic model:", class_models["pathogenic"]["covariance_type"],
      "K=", class_models["pathogenic"]["K"])


In [ ]:
# RELOAD THE SAVED GMM AND GENERATE ONE HELD-OUT PREDICTION

loaded_bundle = joblib.load(demo_model_path)

# Use one variant from the held-out portion of the same fold.
demo_variant = X_raw[demo_fold["test"][0:1]]
demo_variant = np.where(
    np.isnan(demo_variant),
    loaded_bundle["impute_median"],
    demo_variant
)

demo_variant_scaled = (
    demo_variant - loaded_bundle["train_mean"]
) / loaded_bundle["train_std"]

benign_info = loaded_bundle["class_models"]["benign"]
pathogenic_info = loaded_bundle["class_models"]["pathogenic"]

benign_log_density = mixture_logpdf(
    demo_variant_scaled,
    benign_info["model"],
    benign_info["covariance_type"]
)

pathogenic_log_density = mixture_logpdf(
    demo_variant_scaled,
    pathogenic_info["model"],
    pathogenic_info["covariance_type"]
)

decision_score = (
    pathogenic_log_density
    - benign_log_density
    + loaded_bundle["log_prior_ratio"]
)

p_pathogenic = expit(decision_score)
predicted_label = int(np.asarray(decision_score).ravel()[0] > 0)

print("Loaded model successfully")
print("Predicted label:", predicted_label)
print("Pathogenic probability:", float(np.asarray(p_pathogenic).ravel()[0]))
